# Distillation Preprocessing (3)

What we do:
1) Combine inputs and user commands into a single dataset
2) Pre-emptively cluster and select items for labeling
3)

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [3]:
from core.types import *
from doom.utils.doom_game_state import DoomGameState
from core.analysis.intent_clustering import embed_intents
from core.analysis.intent_clustering import cluster_intents
from pathlib import Path

import numpy as np

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
# Load the pre-processed game states
game_states = GameStateEntry.load_states(
    path=Path("data/gamestates/perturbated-gamestates.json"),
    gstype=DoomGameState
)

# Load the generated user commands
commands = UserCommandEntry.load_commands(
    path=Path("data/usercommands/user-commands.json")
)

In [6]:
# Build the dataset of <Game State, User Command> pairs

game_states_map = {
    gse.id: gse
    for gse in game_states
}

inputs = [
    LLMCommandingInput(
        id=command.id,
        game_state=game_states_map[command.state_id],
        user_command=command
    )
    for command in commands
]

print(f"{len(inputs)} game commands generated")

2872 game commands generated


In [7]:
input_lookup = {inp.id: inp for inp in inputs}
intent_list = [inp.user_command.command.intent for inp in inputs]
n_clusters = 10
selected_per_cluster = 5

embeddings = embed_intents(intent_list)
cluster_ids = cluster_intents(embeddings, n_clusters=n_clusters)

# Add cluster_id to inputs
for idx, cluster_id in enumerate(cluster_ids):
    inputs[idx].cluster_id = int(cluster_id)

for inp in inputs:
    inp.selected_for_labelling = False

# Mark inputs for labeling
for cluster_id in range(n_clusters):
    # Get all inputs in this cluster
    clustered_inputs = [inp for inp in inputs if inp.cluster_id == cluster_id]

    if len(clustered_inputs) <= selected_per_cluster:
        for inp in clustered_inputs:
            inp.selected_for_labelling = True
    else:
        sampled_inputs = np.random.choice(
            clustered_inputs,
            size=selected_per_cluster,
            replace=False
        )
        for inp in sampled_inputs:
            inp.selected_for_labelling = True

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

In [8]:
# Save inputs
LLMCommandingInput.save_inputs(
    x=inputs,
    path=Path("data/inputs/inputs.json")
)